[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/surface_curvature/surface_curvature.ipynb)

# Curvature of Quadric Surfaces

How a surface bends at a point is measured by two quadratic forms on its tangent directions: the second fundamental form, which says how fast the surface falls away from its tangent plane, and the first, the ordinary metric. Solved against each other, they give the principal curvatures and directions. This notebook builds both forms for quadric surfaces in PGA3D, then draws the lines of curvature, the curves that follow the principal directions, as an implicit render.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import PGA3D
from examples.geometry.surface_curvature import render
from examples.geometry.surface_curvature.core import direction, nearest_root, pair, point, view_rays

np.set_printoptions(precision=4, suppress=True)

ga = PGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Point = ga.gatype.antivector()                      # finite points, and directions (points with no weight)
Plane = ga.gatype.vector()
Direction = ga.gatype(ga.subspace.antivector().degenerate())   # the ideal points: directions
Quadric = ga.gatype((Plane, Point))                 # a map from points to planes

axes = mv("x y z", np.eye(3))                       # [3] Plane: the coordinate planes x = 0, y = 0, z = 0
w = mv.w                                            # the plane at infinity
print(Direction, Quadric, sep="\n")

## 1. A quadric is a map from points to planes

Writing a type such as `Direction` where a value would go leaves that argument open: the result is an extensor, a linear map still waiting for its input. `axes & Point` is the map that reads a point's coordinates; multiplied by the planes again and summed, the plane dyads build a quadric, `Plane <- Point`, that sends each point to its polar plane. A point lies on the surface when it lies on its own polar plane, `p & Q(p) == 0`. To render it, bind a ray into both slots of that form: it becomes a quadratic in the distance along the ray, and each pixel's ray meets the surface at its smaller root.

In homogeneous coordinates the quadric reads as a symmetric 4×4 matrix $Q$, the surface as $\mathbf x^\top Q\, \mathbf x = 0$, and a point's polar plane as $Q\mathbf x$.

In [ ]:
def quadric(weights: np.ndarray) -> Quadric:
    # One weighted plane dyad per axis, less the dyad of the plane at infinity.
    return (axes * (axes & Point) * weights).sum() - w * (w & Point)


a, b, c = 3.0, 2.0, 1.2
ellipsoid = quadric(1 / np.array([a**2, b**2, c**2]))
hyperboloid = quadric(1 / np.array([1.6**2, 1.0**2, -(1.0**2)]))    # one negative weight: a saddle


def hit(surface: Quadric, origins: Point, heading: Direction) -> tuple[Point, Scalar]:
    # The first hit of each ray, and the discriminant: negative where the ray misses.
    form = Point & surface(Point)                   # Scalar <- (Point, Point): two open slots, a bilinear form
    # Bound to a ray in both slots, the form is a quadratic in the distance along it; its coefficients:
    a, b, c = form(heading, heading), form(heading, origins), form(origins, origins)   # [...] Scalar each
    discriminant = b * b - a * c                   # negative: the ray passes the surface by
    # Step along the ray to the nearer root; adding a direction to a point moves it.
    return (origins + heading * nearest_root(a, b, discriminant)).normalized(), discriminant


cameras = (((25, -60), 3.4), ((18, -60), 3.6))      # (elevation, azimuth) in degrees, and half the image width
rays = [view_rays(*view, extent, 360) for view, extent in cameras]   # [pixels, pixels] origins, one heading per camera
hits = [hit(surface, *ray) for surface, ray in zip((ellipsoid, hyperboloid), rays)]
# What the renderer needs per surface: the hits, which pixels hit, the tangent planes for lighting, the heading.
scenes = [(h, d, surface(h), heading) for surface, (h, d), (_, heading) in zip((ellipsoid, hyperboloid), hits, rays)]
heights = (np.inf, 2.0)                             # the hyperboloid runs to infinity; draw it up to |z| = 2
render.draw_surfaces(scenes, heights);

## 2. The second form against the first

On directions the same map is the surface's Hessian: `Point & Q(Point)`, read on ideal points, says how the quadric's value changes to second order along a direction. The metric on directions is the inner product of their duals, the planes through the origin they are normal to. At a point, the tangent plane is its polar plane, and projecting onto it is a map `Direction <- Direction`. Binding that projector into both slots of the Hessian, and dividing by the gradient's length, gives the second fundamental form. Its eigenvalues against the metric are the principal curvatures, and one more, zero, for the normal, which does not curve. Curvature is measured on directions because that is where PGA's metric lives: it does not see a point's weight.

In the notation of differential geometry the principal curvatures read as the roots of $\det(\mathrm{II} - \kappa\, \mathrm{I}) = 0$, the eigenvalues of the shape operator $\mathrm{I}^{-1}\mathrm{II}$, and the Gaussian curvature as $K = \det\mathrm{II} / \det\mathrm{I}$.

In [ ]:
# A direction's dual is the plane through the origin it is normal to; their inner product is the metric.
metric = Direction.dual() | Direction.dual()       # Scalar <- (Direction, Direction): the first fundamental form


def principal(surface: Quadric, points: Point) -> Scalar:
    # The two principal curvatures at each point, in order.
    tangent = surface(points)                      # the polar plane of a point on the surface is its tangent plane
    normal = tangent.dual().cast(Direction)        # a plane's dual, less its weight: its normal
    project = Direction - normal * ((tangent & Direction) / (tangent & normal))   # onto the tangent plane
    # The Hessian with the projector bound into both slots, over the gradient's length: the second form.
    second = -(Point & surface(Point))(project, project) / metric(normal, normal).square_root()
    # Against the metric: the two curvatures, and the normal's zero, which pair drops.
    return pair(second.eigvalsh(metric))


curvatures = [principal(surface, h) for surface, (h, _) in zip((ellipsoid, hyperboloid), hits)]   # [pixels, pixels, principal] Scalar each
render.draw_gaussian_curvature(scenes, curvatures, heights, 1.2);   # red: both curve the same way; blue: a saddle

In [ ]:
tip = principal(ellipsoid, point(np.array([a, 0.0, 0.0])))
print("curvatures at the tip of the long axis:", tip.to_array(), "  a/b², a/c²:", [a / b**2, a / c**2])

## 3. Lines of curvature from the confocal family

A central quadric sits in a family of confocal quadrics, the quadrics that share its foci, and its lines of curvature are exactly where it meets them. In extensor form the family is a shift: the dual quadric read on directions is the quadric's shape, and each member shifts that shape by a multiple of the metric. A point lies on a member where the shape minus the point's own dyad is singular against the metric, so the members through a point are the eigenvalues of that one form. One is the surface itself; the level sets of the other two draw the net.

In Cartesian coordinates the confocal family of an ellipsoid reads as $\frac{x^2}{a^2 - \lambda} + \frac{y^2}{b^2 - \lambda} + \frac{z^2}{c^2 - \lambda} = 1$, and the three values of $\lambda$ through a point as its ellipsoidal coordinates.

In [ ]:
def confocal(surface: Quadric, points: Point) -> Scalar:
    dual_surface = surface.inverse()                       # Point <- Plane: each plane to its pole, the dual quadric
    pole = dual_surface(w)                                 # the pole of the plane at infinity is the centre
    centre = pole / (w & pole)                     # at unit weight: divide by the pairing with the plane at infinity
    # The plane through the centre normal to each direction, its pole read as a direction, and the
    # plane that direction is normal to: the quadric's shape, a map on directions.
    through_centre = Direction.dual() - w * (Direction.dual() & centre)    # Plane <- Direction
    shape = dual_surface(through_centre).dual()                                    # Plane <- Direction
    position = (points - centre).dual()                                    # [...] Plane: each point's offset from the centre
    # The shape less the point's own dyad; each confocal member shifts it by a multiple of the metric,
    # and the members through the point are the shifts that make it singular: its eigenvalues. One is
    # the surface itself, zero, which pair drops.
    through_point = Direction & (shape - position * (position & Direction))    # [...] Scalar <- (Direction, Direction)
    return pair(through_point.eigvalsh(metric))                                # [..., members] Scalar


parameters = [confocal(surface, h) for surface, (h, _) in zip((ellipsoid, hyperboloid), hits)]   # [pixels, pixels, members] Scalar each
render.draw_curvature_lines(scenes, curvatures, parameters, heights, 1.2);

The four points where the lines on the ellipsoid swirl together are its umbilics: there the surface curves equally in every direction, the two principal curvatures agree, and the two confocal parameters meet. They lie in the plane of the longest and shortest axes.

In [ ]:
# The umbilics' textbook position, in the plane of the longest and shortest axes:
x, z = a * np.sqrt((a**2 - b**2) / (a**2 - c**2)), c * np.sqrt((b**2 - c**2) / (a**2 - c**2))
umbilics = point(np.array([[x, 0, z], [-x, 0, z], [x, 0, -z], [-x, 0, -z]]))   # [umbilics] Point
at_umbilics = principal(ellipsoid, umbilics)
print("principal curvatures at the umbilics:\n", at_umbilics.to_array())
print("confocal parameters there:\n", confocal(ellipsoid, umbilics).to_array())

In [ ]:
# checks
np.testing.assert_allclose(tip.to_array(), [a / b**2, a / c**2], rtol=1e-8)
np.testing.assert_allclose(at_umbilics[:, 0].to_array(), at_umbilics[:, 1].to_array(), rtol=1e-8)
meeting = confocal(ellipsoid, umbilics)
np.testing.assert_allclose(meeting[:, 0].to_array(), meeting[:, 1].to_array(), rtol=1e-6)   # the net closes up at the umbilics